# SERP.fyi FAQ Engine: Dataset Walkthrough

This notebook explores the **40 official SERP.fyi FAQs** and builds the same question-matching engine used by the [SERP.fyi FAQ Engine](https://github.com/SERP-fyi/serp-faq-engine).

- Website: https://serp.fyi
- NPM: `npm install @serp-fyi/serp-faq-engine`
- PyPI: `pip install serp-faq-engine`

## 1. Load the dataset

In [ ]:
import os, re
import pandas as pd

CANDIDATES = [
    "/kaggle/input/serp-faq-dataset/serp_faqs.csv",
    "../dataset/serp_faqs.csv",
    "dataset/serp_faqs.csv",
    "serp_faqs.csv",
]
path = next(p for p in CANDIDATES if os.path.exists(p))
df = pd.read_csv(path)
print(f"Loaded {len(df)} FAQs from {path}")
df.head()

## 2. FAQs by category

In [ ]:
df.groupby("category").size().sort_values(ascending=False).rename("faqs").to_frame()

In [ ]:
df.groupby("part").size().rename("faqs").to_frame()

## 3. The matching engine

Each question is tokenized, stopwords are removed, plurals are singularised and common synonyms are mapped (for example *rankings* to *rank* or *ChatGPT* to *AI*). Every FAQ is scored by weighted keyword overlap: question words count 2.0, keywords 1.5 and answer words 0.5. Confidence is the score as a percentage of a perfect question match.

In [ ]:
STOPWORDS = set(
    "a an the is are was were be been do does did can could will would should "
    "i me my we our you your it its to of in on for with and or at by from as "
    "how what which who when where why this that there their them they if so "
    "any get got have has about into up out than then just very need anything".split()
)
SYNONYMS = {
    "ranking": "rank", "rankings": "rank", "ranked": "rank", "position": "rank", "positions": "rank",
    "chatgpt": "ai", "llm": "ai", "llms": "ai", "gemini": "ai", "perplexity": "ai", "claude": "ai",
    "aivo": "ai", "geo": "ai", "generative": "ai", "aio": "overview",
    "price": "cost", "pricing": "cost", "prices": "cost",
    "private": "privacy", "confidential": "privacy", "secure": "privacy", "security": "privacy",
    "download": "export", "downloading": "export", "exporting": "export",
    "rival": "competitor", "rivals": "competitor",
    "phone": "mobile", "smartphone": "mobile",
    "maps": "local", "map": "local", "gbp": "local",
    "fresh": "live", "realtime": "live", "update": "refresh", "updated": "refresh", "updates": "refresh",
    "signup": "register", "sign": "register", "begin": "start", "started": "start",
    "help": "support", "contact": "support",
    "installation": "install", "installed": "install",
    "whitelabel": "white", "rebrand": "white",
    "made": "created", "developed": "created", "founded": "created", "creator": "created",
    "mentioned": "mention", "cited": "citation", "cite": "citation",
}
WEIGHTS = {"question": 2.0, "keywords": 1.5, "answer": 0.5}

def tokenize(text):
    out = []
    for w in re.split(r"[^a-z0-9]+", str(text).lower().replace("serp.fyi", "serpfyi")):
        if not w or w in STOPWORDS:
            continue
        w = SYNONYMS.get(w, w)
        if len(w) > 3 and w.endswith("s") and not w.endswith("ss"):
            w = w[:-1]
        out.append(SYNONYMS.get(w, w))
    return out

INDEX = []
for row in df.itertuples():
    idx = {}
    for field, weight in WEIGHTS.items():
        for t in tokenize(getattr(row, field)):
            idx[t] = max(idx.get(t, 0.0), weight)
    INDEX.append((idx, row))

def band(score):
    return "Strong match" if score > 80 else "Good match" if score > 60 else "Possible match" if score > 30 else "No match"

def search(query, top=3):
    q = list(dict.fromkeys(tokenize(query)))
    rows = []
    for idx, row in INDEX:
        points = sum(idx.get(t, 0.0) for t in q)
        if points > 0:
            conf = min(100, int(points / (2.0 * len(q)) * 100 + 0.5))
            rows.append({"id": row.id, "question": row.question, "answer": row.answer, "confidence": conf, "band": band(conf)})
    rows.sort(key=lambda r: (-r["confidence"], r["id"]))
    return pd.DataFrame(rows[:top])

## 4. Ask questions

In [ ]:
search("does serp.fyi track chatgpt mentions")

In [ ]:
search("can I track mobile rankings")

In [ ]:
search("can agencies white label reports", top=1)

## 5. Self-test: every FAQ question should find itself

In [ ]:
hits = sum(int(search(q, 1).iloc[0]["id"]) == i for i, q in zip(df["id"], df["question"]))
print(f"{hits}/{len(df)} FAQ questions matched their own answer")

## Links

| Platform | URL |
|---|---|
| Website | https://serp.fyi |
| GitHub | https://github.com/SERP-fyi/serp-faq-engine |
| NPM | https://www.npmjs.com/package/@serp-fyi/serp-faq-engine |
| PyPI | https://pypi.org/project/serp-faq-engine/ |
| Hugging Face | https://huggingface.co/datasets/serp-fyi/serp-faq-dataset |
| Docs | https://serp-faq-engine.readthedocs.io |